In [70]:
import os
import re

os.environ["HF_API_TOKEN"] = ""

In [71]:
!pip install -q youtube-transcript-api langchain-community langchain-huggingface langchain-text-splitters \
               faiss-cpu tiktoken python-dotenv

In [72]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [73]:
def get_youtube_video_id(url):
    pattern = r'(?:youtube\.com/(?:watch\?v=|embed/|shorts/)|youtu\.be/)([^&?/]+)'
    match = re.search(pattern, url)
    return match.group(1) if match else None

In [74]:
video_url = input("Enter the YouTube video URL: ")

In [75]:
video_id = get_youtube_video_id(video_url)
ytt_api = YouTubeTranscriptApi()

try:
    transcript_language_list  = ytt_api.list(video_id)
    transcript_language = (next(iter(transcript_language_list))).language_code

    # Use YouTubeTranscriptApi.get_transcript to directly get the list of dictionaries
    transcript_list = ytt_api.fetch(video_id, languages = [transcript_language])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)

except TranscriptsDisabled:
    print("No captions available for this video.")

In [76]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [77]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
)

vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [78]:
llm = HuggingFaceEndpoint(
    huggingfacehub_api_token=os.environ["HF_API_TOKEN"],
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    temperature=0.7,
    task="conversational",
    max_new_tokens=1000,
)

model = ChatHuggingFace(llm = llm)

In [79]:
question = input("Enter your question: ")

Enter your question: summerize the video


In [80]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [81]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

retrieved_docs = retriever.invoke(question)

In [82]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [83]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [84]:
parser = StrOutputParser()

In [85]:
main_chain = parallel_chain | prompt | model | parser

In [86]:
main_chain.invoke(question)

'The video discusses the importance of taking action and making decisions when it comes to startup ideas. It suggests that instead of trying to find the perfect idea, one should pick an idea and commit to it fully. This involves burning other ideas and devoting time and resources to learning about the customer and executing on the chosen idea.\n\nThe video emphasizes the importance of "going deep" and becoming an expert in the chosen domain. It provides examples of companies that have successfully pivoted and changed their focus to achieve success.\n\nThe video highlights three key qualities of a good idea:\n\n1. It should be the direction you\'re committed to, with a willingness to change your approach as you learn.\n2. It should be the most ambitious version of itself, pushing the boundaries of what\'s possible.\n3. It should be the only thing you\'re working on, with a focus on execution and learning from the customer.'